In [10]:
"""
Nombre: Almaraz Hernández María de los Ángeles
"""

import numpy as np
import pandas as pd
from google.colab import drive

class Grafica():
    """Representa mi gráfica con nodos y arcos."""
    def __init__(self, nodos, arcos):
        """Definicion de mi constructor."""
        self.matriz = np.zeros((len(nodos), len(nodos)))
        for x in arcos:
            """Definimos mi coordenada como 1 para que
            se agregue a mi matriz vacia"""
            self.matriz[x[0], x[1]] = 1

    def __str__(self):
        """Enseñamos la matriz"""
        return str(self.matriz)

    def agregar_arcos(self, tupla):
        """Agregamos un arco a nuestra grafica"""
        self.matriz[tupla[0], tupla[1]] = 1

    def quitar_arcos(self, tupla):
        """Quitamos un arco a nuestra gráfica"""
        self.matriz[tupla[0], tupla[1]] = 0

    def num_nodos(self):
        """Verificamos el num de nodos"""
        return len(self.matriz)

    def num_arcos(self):
        """Verificamos el num de arcos"""
        return self.matriz.sum()

    def agregar_nodo(self):
        """Agregamos nodos"""
        vertical = np.zeros((len(self.matriz),1))
        horizontal = np.zeros((1,len(self.matriz)+1))
        self.matriz = np.vstack((np.hstack((self.matriz, vertical)),
                             horizontal))

    def eliminar_nodo(self):
        """Eliminamos nodos"""
        self.matriz = np.delete(self.matriz,(nodo),axis = 0)
        self.matriz = np.delete(self.matriz,(nodo),axis = 1)

class Red(Grafica):
    """Red hereda de Grafica"""
    def __init__(self, nodos, arcos):
        """Constructor"""
        super().__init__(nodos, arcos)

    def foomatrix1(self):
        """Mis columnas sumaran 1"""
        foo = np.zeros_like(self.matriz)
        for i in range(len(self.matriz)):
            foo[i,:] = self.matriz[i,:] / self.matriz[i,:].sum()
        return foo

def construccion_pd(link_archivo):
    """Hace que mi archivo se lea como una tabla"""
    df = pd.read_excel(link_archivo)
    nodos = (list(range(len(df["Index"]))))
    arcos = []
    for i, cited_by in enumerate(df["Cited by"]):
        for j in cited_by.split(','):
            arcos.append((int(j.strip()) - 1, i))
    return Red(nodos, arcos)

def foomultiplicador(matrix):
    """Vector de excel, matrix = matriz archivo"""
    foo = np.zeros_like(matrix)
    for i in range(len(matrix)):
        foo[i,:] = matrix[i,:] / matrix[i,:].sum()
    pi = np.ones((1,len(matrix))) / len(matrix)
    pi_1 = pi@foo
    while np.linalg.norm(pi - pi_1) > 1e-6:
      """Mientras la norma sea mayor a 1e-6"""
      pi = pi_1
      pi_1 = pi@foo
      pi.sum()
    return pi

def foo_ru(matrix, dominio, d):
    """Vector de excel con .ru"""
    n = len(matrix)
    indices_ru = []
    for i, dominio in enumerate(dominios):
      """Ver si cada indice termina en .ru, se agrega a indices_ru"""
      if dominio.endswith(".ru"):
        indices_ru.append(i)
    """Veremos si hay pag .ru"""
    p = len(indices_ru)
    pagerank = np.zeros(n)
    if not indices_ru:
        raise ValueError("No hay paginas .ru")
    """Vector pi"""
    pi = np.zeros(n)
    for indices in indices_ru:
        """Para cada indice en indices_ru"""
        pi[indices] = 1 / p
    """Matriz U"""
    U = np.ones((n, n)) / n
    for x in range(100):
        pi_ru = d * matrix.T@pi + (1 - d) * (U@pi)
        while np.linalg.norm(pi - pi_ru) > 1e-6:
            return pi_ru
    """Regresa el vector despues de 100 iteraciones"""
    return pi

if __name__ == "__main__":
    """Google drive, ruta del archivo, construir mi
     matriz y mi red, matriz de red y mi foo multiplicador"""
    drive.mount("mnt")
    link_archivo = "/content/mnt/MyDrive/Web.xlsx"
    matrix = construccion_pd(link_archivo).matriz
    red = construccion_pd(link_archivo)
    matrix_foo = red.foomatrix1()
    pagerank = foomultiplicador(matrix_foo)

    """1"""
    print(f"PageRank1 básico: {pagerank}")
    print("LA PAGINA WEB MAS IMPORTANTE ES LA NUM 26")

    """2 Y 3"""
    dominios = pd.read_excel(link_archivo)["Website"].tolist()
    pagerank_ru1 = foo_ru(matrix_foo, dominios, d=0.5)
    pagerank_ru2 = foo_ru(matrix_foo, dominios, d=0.85)
    pagerank_ru3 = foo_ru(matrix_foo, dominios, d=1)

    print(f"PageRank1 con .ru: {pagerank_ru1}")
    print(f"PageRank2 con .ru: {pagerank_ru2}")
    print(f"PageRank3 con .ru: {pagerank_ru3}")

    print("LA PAGINA WEB NUM 26 YA NO ES LA MÁS IMPORTANTE")
    print("""EL VALOR 1 YA QUE ES MÁS FACIL SABER
          CUALES SON LAS PAGINAS RU MÁS FRECUENTADAS""")

Drive already mounted at mnt; to attempt to forcibly remount, call drive.mount("mnt", force_remount=True).
PageRank1 básico: [[7.33320314e-07 1.01764169e-06 6.33367242e-07 1.36085707e-06
  5.34701306e-07 8.20661192e-07 1.37005441e-06 8.33927232e-07
  1.10169011e-06 9.60685274e-07 5.04500778e-07 5.24041574e-07
  7.54179322e-07 7.57679334e-07 9.73391912e-07 3.60576923e-02
  3.60576923e-02 3.60576923e-02 3.60576923e-02 4.80769231e-02
  3.60576923e-02 3.60576923e-02 3.60576923e-02 4.80769231e-02
  3.60576923e-02 6.15371735e-01]]
LA PAGINA WEB MAS IMPORTANTE ES LA NUM 26
PageRank1 con .ru: [0.01923077 0.01923077 0.01923077 0.01923077 0.01923077 0.01923077
 0.01923077 0.01923077 0.01923077 0.01923077 0.01923077 0.01923077
 0.01923077 0.01923077 0.01923077 0.0650641  0.06923077 0.0650641
 0.0650641  0.08589744 0.0650641  0.0650641  0.06089744 0.08589744
 0.0650641  0.01923077]
PageRank2 con .ru: [0.00576923 0.00576923 0.00576923 0.00576923 0.00576923 0.00576923
 0.00576923 0.00576923 0.005769